# ARC NeuroGolf static ONNX solver 09- localish_recolor

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
TASK_ID='task340'
CH=10
H=W=30
MODEL_VERSION='task340-edge-projection-static-graph'

In [4]:
import importlib.util, subprocess, sys
required={'onnx':'onnx','onnxruntime':'onnxruntime','torch':'torch','numpy':'numpy'}
missing=[pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])


import json, os, random, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 41.7 MB/s eta 0:00:00


In [5]:
import json, random, zipfile
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import onnx, onnxruntime as ort

class EdgeProjectionStatic(nn.Module):
    def forward(self, x):
        active = (x.sum(1, keepdim=True) > 0).float()
        row_valid = (active.sum(3, keepdim=True) > 0).float()
        col_valid = (active.sum(2, keepdim=True) > 0).float()
        h_eff = row_valid.sum((2,3), keepdim=True)
        w_eff = col_valid.sum((2,3), keepdim=True)
        out_ch = []
        occupied = active * 0
        for k in range(1, CH):
            m = x[:, k:k+1]
            row_count = m.sum(3, keepdim=True)
            col_count = m.sum(2, keepdim=True)
            long_row = (row_count >= torch.clamp(w_eff - 2.0, min=3.0)).float() * row_valid
            long_col = (col_count >= torch.clamp(h_eff - 2.0, min=3.0)).float() * col_valid
            base = m * ((long_row > 0).float() + (long_col > 0).float()).clamp(0, 1)
            markers = m * (1 - base)
            above = torch.cumsum(markers, dim=2)
            below = torch.flip(torch.cumsum(torch.flip(markers, dims=[2]), dim=2), dims=[2])
            left = torch.cumsum(markers, dim=3)
            right = torch.flip(torch.cumsum(torch.flip(markers, dims=[3]), dim=3), dims=[3])
            zrow = long_row[:, :, :1, :] * 0
            zcol = long_col[:, :, :, :1] * 0
            long_row_below = torch.cat([long_row[:, :, 1:, :], zrow], dim=2)
            long_row_above = torch.cat([zrow, long_row[:, :, :H-1, :]], dim=2)
            long_col_right = torch.cat([long_col[:, :, :, 1:], zcol], dim=3)
            long_col_left = torch.cat([zcol, long_col[:, :, :, :W-1]], dim=3)
            proj_up = long_row_below * (above > 0).float()
            proj_down = long_row_above * (below > 0).float()
            proj_left = long_col_right * (left > 0).float()
            proj_right = long_col_left * (right > 0).float()
            col = (base + proj_up + proj_down + proj_left + proj_right).clamp(0, 1) * active
            out_ch.append(col)
            occupied = (occupied + col).clamp(0, 1)
        bg = (1 - occupied) * active
        return torch.cat([bg] + out_ch, 1)

In [6]:
def load_task(task_id):
    for p in [Path.cwd()/f'{task_id}.json', 
              Path.cwd().parent/f'{task_id}.json', 
              Path('/mnt/data')/f'{task_id}.json',
              Path(COMPETITION)/f'{task_id}.json']:
        if p.exists():
            return json.load(open(p)), p
    raise FileNotFoundError(task_id)

def onehot(grid):
    arr=np.array(grid,dtype=np.int64); h,w=arr.shape
    x=np.zeros((1,CH,H,W),dtype=np.float32)
    for k in range(CH): x[0,k,:h,:w]=(arr==k)
    return x,h,w

task, task_path = load_task(TASK_ID)
ROOT=Path.cwd(); OUT_DIR=ROOT/'generated_models'; OUT_DIR.mkdir(exist_ok=True)
MODEL_PATH=OUT_DIR/f'{TASK_ID}.onnx'
print(task_path, {k:len(v) for k,v in task.items() if isinstance(v,list)})

/kaggle/input/competitions/neurogolf-2026/task340.json {'train': 3, 'test': 1, 'arc-gen': 262}


In [7]:
model=EdgeProjectionStatic().eval()
torch.onnx.export(model, torch.zeros(1,CH,H,W), str(MODEL_PATH), input_names=['input'], output_names=['output'], opset_version=13, dynamo=False)
m=onnx.load(str(MODEL_PATH)); m.ir_version=min(m.ir_version,7); onnx.save(m,str(MODEL_PATH))
print('exported', MODEL_PATH, MODEL_PATH.stat().st_size)

/tmp/ipykernel_16/753512908.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, torch.zeros(1,CH,H,W), str(MODEL_PATH), input_names=['input'], output_names=['output'], opset_version=13, dynamo=False)


exported /kaggle/working/generated_models/task340.onnx 96097


In [8]:
def inspect(model_path):
    m=onnx.load(str(model_path)); ops={}
    for n in m.graph.node: ops[n.op_type]=ops.get(n.op_type,0)+1
    return {'size':model_path.stat().st_size,'input_shape':[d.dim_value for d in m.graph.input[0].type.tensor_type.shape.dim], 'output_shape':[d.dim_value for d in m.graph.output[0].type.tensor_type.shape.dim], 'forbidden':[op for op in ['Loop','Scan','NonZero','Unique','Script','Function'] if ops.get(op,0)], 'dynamic_risk_ops':[op for op in ['Shape','Range','Expand','ScatterND','Gather','ConstantOfShape','Resize','NonZero'] if ops.get(op,0)], 'op_counts':ops}

def validate(model_path):
    sess=ort.InferenceSession(str(model_path), providers=['CPUExecutionProvider']); rep={}
    for sec in ['train','test','arc-gen']:
        exact=0; wrong=0
        for ex in task[sec]:
            x,h,w=onehot(ex['input']); pred=sess.run(None, {'input':x})[0].argmax(1)[0,:h,:w]; out=np.array(ex['output'])
            exact += int(np.array_equal(pred,out)); wrong += int((pred!=out).sum())
        rep[sec]={'exact':exact,'total':len(task[sec]),'wrong_pixels':wrong}
    return rep
arch=inspect(MODEL_PATH); val=validate(MODEL_PATH)
print(json.dumps(arch, indent=2)); print(json.dumps(val, indent=2))
assert arch['input_shape']==[1,10,30,30] and arch['output_shape']==[1,10,30,30]
assert arch['size'] < 1_400_000
assert not arch['forbidden']
assert not arch['dynamic_risk_ops']
assert val['train']['exact']==val['train']['total']
assert val['test']['exact']==val['test']['total']
assert val['arc-gen']['exact']==val['arc-gen']['total']

{
  "size": 96097,
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "forbidden": [],
  "dynamic_risk_ops": [],
  "op_counts": {
    "Constant": 580,
    "ReduceSum": 23,
    "Greater": 57,
    "Cast": 75,
    "Mul": 101,
    "Slice": 99,
    "Sub": 12,
    "Clip": 29,
    "GreaterOrEqual": 18,
    "Add": 54,
    "CumSum": 36,
    "Concat": 37
  }
}
{
  "train": {
    "exact": 3,
    "total": 3,
    "wrong_pixels": 0
  },
  "test": {
    "exact": 1,
    "total": 1,
    "wrong_pixels": 0
  },
  "arc-gen": {
    "exact": 262,
    "total": 262,
    "wrong_pixels": 0
  }
}


In [9]:
idx=list(range(len(task['arc-gen']))); random.Random(0).shuffle(idx); test=set(idx[:round(len(idx)*0.3)])
sess=ort.InferenceSession(str(MODEL_PATH), providers=['CPUExecutionProvider'])
fe=ft=te=tt=0
for i,ex in enumerate(task['arc-gen']):
    x,h,w=onehot(ex['input']); pred=sess.run(None, {'input':x})[0].argmax(1)[0,:h,:w]
    ok=int(np.array_equal(pred,np.array(ex['output'])))
    if i in test: te+=ok; tt+=1
    else: fe+=ok; ft+=1
print({'fit_exact':fe,'fit_total':ft,'test_exact':te,'test_total':tt})
assert te==tt

{'fit_exact': 183, 'fit_total': 183, 'test_exact': 79, 'test_total': 79}


In [10]:
zip_path=ROOT/'submission.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    zf.write(MODEL_PATH, MODEL_PATH.name)
print('submission:', zip_path, zipfile.ZipFile(zip_path).namelist())

submission: /kaggle/working/submission.zip ['task340.onnx']
